# CSV-first 国债期货数据管线

所有 Wind/手工输入先生成可审计 CSV，再增量同步 DuckDB。首次建库仍使用 `T_mindf.csv`；日常更新按日线 5 个交易日、分钟线 2 个交易日回溯覆盖。

In [ ]:
from pathlib import Path
import sys

# 自动向上寻找项目根目录，避免 Notebook 从不同工作目录启动时路径失效。
_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in _PROJECT_CANDIDATES if (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到项目根目录 pyproject.toml')

# 将 src 加入 Python 搜索路径，从而直接导入本项目的 datapipeline 包。
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from IPython.display import display
import duckdb
import pandas as pd

# CSV-first 管线的五个主要入口：经济日历、全量、增量和两类补丁。
from datapipeline.eco_calendar_pipeline import build_eco_calendar_files
from datapipeline.pipeline_manager import (
    run_contract_wind_patch,
    run_full_initialization,
    run_historical_minute_patch,
    run_incremental_update,
)

# 所有目录都由 paths.py 统一维护，Notebook 中不再硬编码绝对路径。
from datapipeline.paths import (
    CSV_DATA_DIR, DATA_PIPELINE_LOG_DIR, DB_PATH, DUCKDB_BACKUP_DIR,
    ECO_CALENDAR_CSV_PATH, ECO_CALENDAR_RAW_DIR, ECO_CALENDAR_XLSX_PATH,
)

## 数据落地位置

- 原始经济日历：`data/csv/eco_calendar/`
- 历史分钟输入：`data/csv/T_mindf.csv`
- filtered Excel：`data/input/eco_calendar_filtered.xlsx`
- 标准 CSV：`data/csv/reference/`、`data/csv/daily/`、`data/csv/minute/`
- DuckDB 备份：`data/backups/duckdb/`
- 每次运行日志：`logs/datapipeline/<run_id>/`

不要同时打开多个运行开关。先只构建经济日历不需要 Wind。

In [ ]:
# ------------------------------ 运行开关 ------------------------------
# 只构建经济日历：不连接 Wind，也不修改 DuckDB。
RUN_ECO_CALENDAR_ONLY = False
# 第一次全量建库：使用原始经济日历、T_mindf.csv 和 Wind；一般只运行一次。
RUN_FULL_INITIALIZATION = False
# 日常增量更新：只更新尚未到期的合约，并按下方天数回溯覆盖。
RUN_INCREMENTAL_UPDATE = False
# 用 T_mindf.csv 修复指定交易日，默认保留原有的 2024-11-21 补丁。
RUN_HISTORICAL_MINUTE_PATCH = False
# 从 Wind 强制重抓某个合约的指定日线或分钟线区间。
RUN_CONTRACT_WIND_PATCH = False
# 只读查看 DuckDB 各表覆盖范围；可以单独打开。
RUN_AUDIT = False

# 防止一次误开多个写入任务。RUN_AUDIT 是只读操作，因此不计入此限制。
enabled = [
    RUN_ECO_CALENDAR_ONLY, RUN_FULL_INITIALIZATION, RUN_INCREMENTAL_UPDATE,
    RUN_HISTORICAL_MINUTE_PATCH, RUN_CONTRACT_WIND_PATCH,
]
if sum(enabled) > 1:
    raise ValueError('一次只允许打开一个写入开关')

# 全量历史起点和本次运行终点。运行较久时，可在开始前重新执行本单元刷新 UPDATE_END。
START_DATE = '2015-06-30'
UPDATE_END = pd.Timestamp.now().floor('s')

# 为吸收 Wind 对近期数据的修订，增量更新会重新下载这些交易日并按主键去重覆盖。
DAILY_OVERLAP_TRADING_DAYS = 5
MINUTE_OVERLAP_TRADING_DAYS = 2

# 历史分钟补丁参数：数据来源为 data/csv/T_mindf.csv。
HISTORICAL_PATCH_DATE = '2024-11-21'

# Wind 补丁参数：dataset 只能选择 daily 或 minute。
WIND_PATCH_DATASET = 'daily'  # 'daily' 或 'minute'
WIND_PATCH_CONTRACT = 'T2609.CFE'
WIND_PATCH_START = '2026-07-01'
WIND_PATCH_END = '2026-07-23 15:15:00'

In [ ]:
# 此模式只合并 data/csv/eco_calendar/ 中的原始月度 Excel。
# 输出 filtered.xlsx 供现有因子流程兼容，同时输出 CSV 供 DuckDB 使用。
if RUN_ECO_CALENDAR_ONLY:
    eco_result = build_eco_calendar_files()
    display(pd.Series(eco_result, name='eco_calendar'))

In [ ]:
# 只有需要调用 Wind 的三种模式才启动 WindPy；历史 CSV 补丁不需要 Wind。
if RUN_FULL_INITIALIZATION or RUN_INCREMENTAL_UPDATE or RUN_CONTRACT_WIND_PATCH:
    from WindPy import w

    w.start(waitTime=60)
    if not w.isconnected():
        raise ConnectionError('WindPy 未连接')

# 首次建库顺序：
# 1) 构建经济日历；2) 保存合约目录和主力映射 CSV；
# 3) 将 T_mindf.csv 拆为分合约分钟 CSV；4) 下载分合约日线/分钟线 CSV；
# 5) 应用历史补丁；6) 备份旧库；7) 从 CSV 构建 DuckDB。
if RUN_FULL_INITIALIZATION:
    full_result = run_full_initialization(
        w, start_date=START_DATE, end_time=UPDATE_END,
        historical_patch_dates=(HISTORICAL_PATCH_DATE,),
    )
    display(pd.Series(full_result, name='full_initialization'))

# 日常增量更新只请求尚未结束的合约；到期合约 CSV 和库内历史数据保持不变。
# DuckDB 只删除并重写本次发生变化的合约区间，不会每天全量重建行情表。
if RUN_INCREMENTAL_UPDATE:
    incremental_result = run_incremental_update(
        w, start_date=START_DATE, end_time=UPDATE_END,
        daily_overlap_trading_days=DAILY_OVERLAP_TRADING_DAYS,
        minute_overlap_trading_days=MINUTE_OVERLAP_TRADING_DAYS,
    )
    display(pd.Series(incremental_result, name='incremental_update'))

In [ ]:
# 历史分钟补丁：先改分合约 CSV，再备份和更新 DuckDB，保留完整审计链路。
if RUN_HISTORICAL_MINUTE_PATCH:
    patch_result = run_historical_minute_patch(HISTORICAL_PATCH_DATE)
    display(pd.Series(patch_result, name='historical_minute_patch'))

# Wind 区间补丁：适用于指定合约的缺失、异常或需要强制覆盖的数据。
# 与正常更新一样，Wind 返回值必须先成功写入 CSV，之后才会同步 DuckDB。
if RUN_CONTRACT_WIND_PATCH:
    wind_patch_result = run_contract_wind_patch(
        w, WIND_PATCH_DATASET, WIND_PATCH_CONTRACT,
        WIND_PATCH_START, WIND_PATCH_END,
    )
    display(pd.Series(wind_patch_result, name='contract_wind_patch'))

In [ ]:
# 只读审计，不修改数据库。用于快速确认经济日历、行情及主连表的覆盖范围。
if RUN_AUDIT:
    if not DB_PATH.is_file():
        raise FileNotFoundError(DB_PATH)
    # 使用 read_only=True，避免审计单元意外取得写锁。
    audit_con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        overview = audit_con.execute(
            '''
            SELECT 'eco_calendar' AS table_name, count(*) AS rows,
                   min(event_datetime)::VARCHAR AS min_time, max(event_datetime)::VARCHAR AS max_time FROM eco_calendar
            UNION ALL SELECT 'daily_bars', count(*), min(trade_date)::VARCHAR, max(trade_date)::VARCHAR FROM daily_bars
            UNION ALL SELECT 'minute_bars', count(*), min(bar_time)::VARCHAR, max(bar_time)::VARCHAR FROM minute_bars
            UNION ALL SELECT 'main_daily_continuous', count(*), min(trade_date)::VARCHAR, max(trade_date)::VARCHAR FROM main_daily_continuous
            UNION ALL SELECT 'main_minute_continuous', count(*), min(bar_time)::VARCHAR, max(bar_time)::VARCHAR FROM main_minute_continuous
            '''
        ).fetchdf()
    finally:
        # 无论查询成功或失败都释放连接，避免影响后续数据更新。
        audit_con.close()
    display(overview)
    print('CSV:', CSV_DATA_DIR)
    print('Backups:', DUCKDB_BACKUP_DIR)
    print('Logs:', DATA_PIPELINE_LOG_DIR)